# Assignment 3 - PCD
Alif Ryanatha Azhar Suryanto - 25/569917/PA/24023

Task: segmentation + morphological operations without using any library function (so no cv2.threshold, cv2.dilate, etc). Then compare the segmentation result before and after morphology is applied.

I made my own test image with numpy instead of uploading a file, easier that way

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(1)

### make a test image (rectangle + circle + noise)

In [ ]:
size = 200
img = np.full((size, size), 40, dtype=np.uint8)

img[30:90, 30:110] = 200

yy, xx = np.ogrid[:size, :size]
circle = (xx-150)**2 + (yy-60)**2 <= 35**2
img[circle] = 210

blob = ((xx-100)/55)**2 + ((yy-150)/30)**2 <= 1
img[blob] = 190

img_clean = img.copy()

# add salt and pepper noise manually
h, w = img.shape
n = int(0.06 * h * w)
for i in range(n//2):
    y, x = np.random.randint(0,h), np.random.randint(0,w)
    img[y,x] = 255
for i in range(n//2):
    y, x = np.random.randint(0,h), np.random.randint(0,w)
    img[y,x] = 0

plt.imshow(img, cmap='gray')
plt.title('test image + noise')
plt.show()

### segmentation using otsu

this picks the threshold automatically, finds the T that gives the biggest between-class variance for foreground-background. I compute the histogram manually then loop through every T from 0-255.

In [ ]:
def otsu(img):
    hist = np.zeros(256)
    for v in img.ravel():
        hist[v] += 1

    total = img.size
    sum_all = sum(i*hist[i] for i in range(256))

    sum_bg = 0
    w_bg = 0
    best_T = 0
    best_var = 0

    for T in range(256):
        w_bg += hist[T]
        if w_bg == 0:
            continue
        w_fg = total - w_bg
        if w_fg == 0:
            break
        sum_bg += T*hist[T]
        m_bg = sum_bg/w_bg
        m_fg = (sum_all - sum_bg)/w_fg
        var_between = w_bg*w_fg*(m_bg-m_fg)**2
        if var_between > best_var:
            best_var = var_between
            best_T = T
    return best_T

T = otsu(img)
print("threshold:", T)

seg = np.zeros_like(img)
seg[img > T] = 1

plt.imshow(seg, cmap='gray')
plt.title(f'segmentation result (T={T}) - no morphology yet')
plt.show()

### manual morphology

structuring element is 3x3 all 1s. dilation = if at least 1 neighbor is foreground then it becomes foreground. erosion = only stays foreground if ALL neighbors are foreground, otherwise becomes background. opening = erode first then dilate. closing = dilate first then erode.

In [ ]:
kernel = np.ones((3,3))

def dilate(bimg):
    padded = np.pad(bimg, 1)
    out = np.zeros_like(bimg)
    for i in range(bimg.shape[0]):
        for j in range(bimg.shape[1]):
            window = padded[i:i+3, j:j+3]
            if window.max() == 1:
                out[i,j] = 1
    return out

def erode(bimg):
    padded = np.pad(bimg, 1)
    out = np.zeros_like(bimg)
    for i in range(bimg.shape[0]):
        for j in range(bimg.shape[1]):
            window = padded[i:i+3, j:j+3]
            if window.min() == 1:
                out[i,j] = 1
    return out

def opening(bimg):
    return dilate(erode(bimg))

def closing(bimg):
    return erode(dilate(bimg))

d = dilate(seg)
e = erode(seg)
o = opening(seg)
c = closing(seg)

fig, ax = plt.subplots(2,2, figsize=(8,8))
ax[0,0].imshow(d, cmap='gray'); ax[0,0].set_title('dilation')
ax[0,1].imshow(e, cmap='gray'); ax[0,1].set_title('erosion')
ax[1,0].imshow(o, cmap='gray'); ax[1,0].set_title('opening')
ax[1,1].imshow(c, cmap='gray'); ax[1,1].set_title('closing')
for a in ax.ravel():
    a.axis('off')
plt.show()

### compare before vs after morphology

the salt and pepper noise creates random white dots in the background and small black holes inside the shapes. so I used opening first (removes the noise dots) then closing (fills the small holes).

In [ ]:
seg_clean = closing(opening(seg))

fig, ax = plt.subplots(1,3, figsize=(14,5))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('input')
ax[1].imshow(seg, cmap='gray'); ax[1].set_title('no morphology')
ax[2].imshow(seg_clean, cmap='gray'); ax[2].set_title('with morphology')
for a in ax:
    a.axis('off')
plt.show()

# check accuracy against ground truth (image without noise)
T2 = otsu(img_clean)
gt = np.zeros_like(img_clean)
gt[img_clean > T2] = 1

acc_before = np.mean(seg == gt) * 100
acc_after = np.mean(seg_clean == gt) * 100

print("accuracy before morphology:", round(acc_before,2), "%")
print("accuracy after morphology:", round(acc_after,2), "%")
print("foreground pixel count before:", int(seg.sum()))
print("foreground pixel count after:", int(seg_clean.sum()))

### conclusion

without morphology the segmentation result still has a lot of noise (random white dots in the background and small holes inside the shapes). after applying opening+closing the noise disappears because it's smaller than the structuring element used, while the actual shapes (rectangle, circle, blob) stay preserved because they're bigger. pixel accuracy compared to the ground truth also goes up after applying morphology.

github link: (fill in before submitting)